# 02 - The dataset and the MONAI transforms

A neural network cannot read a NIfTI file. In this notebook we prepare the images for the network with
**MONAI**, a library for deep learning on medical images built on top of PyTorch.

You only need to run the cells and read. The exercises are at the end.

## Index

- [1. Setup](#1.-Setup)
- [2. The dataset](#2.-The-dataset)
- [3. What the network needs](#3.-What-the-network-needs)
- [4. Loading the files](#4.-Loading-the-files)
- [5. Adding the channel](#5.-Adding-the-channel)
- [6. Keeping only the lungs](#6.-Keeping-only-the-lungs)
- [7. Rescaling the intensities](#7.-Rescaling-the-intensities)
- [8. Resizing](#8.-Resizing)
- [9. Compose: all the steps together](#9.-Compose:-all-the-steps-together)
- [10. Data augmentation](#10.-Data-augmentation)
- [11. Exercises](#11.-Exercises)
- [12. To go further](#12.-To-go-further)

## 1. Setup

In [ ]:
!pip install -q monai

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from monai.transforms import (
    Compose,
    EnsureChannelFirstd,
    Lambdad,
    LoadImaged,
    RandAffined,
    RandFlipd,
    RandGaussianNoised,
    Resized,
    ScaleIntensityRanged,
)

In [ ]:
# Folder of the course dataset (on Kaggle: right panel -> Input -> Add Input)
# On Kaggle the datasets added with "Add Input" are under /kaggle/input, in a subfolder
# (datasets/<owner>/). We look for the course dataset by its name.
DATA_DIR = next(Path("/kaggle/input").rglob("luna16-lung-slices"), Path("/kaggle/input/luna16-lung-slices"))

## 2. The dataset

`metadata.csv` has one row per patient. Remember: one patient = one CT slice = one image and one mask.

In [ ]:
metadata = pd.read_csv(DATA_DIR / "metadata.csv")
print("number of patients:", len(metadata))
metadata[["case_id", "spacing_x", "slice_thickness", "z_rel", "lung_area_mm2"]].head()

- `spacing_x`: size of a pixel in mm. It is different from patient to patient.
- `z_rel`: position of the slice along the body, from 0 (bottom of the scan) to 1 (top). The slices were taken at
  random heights, so some show large lungs and some only the top or the bottom of them.

The files have the same name in the two folders. We build a list with one **dictionary** per patient:
the path of the image and the path of the mask.

In [ ]:
image_paths = sorted((DATA_DIR / "images").glob("*.nii"))
label_paths = sorted((DATA_DIR / "labels").glob("*.nii"))

dataset_files = []
for image_path, label_path in zip(image_paths, label_paths):
    dataset_files.append({"image": image_path, "label": label_path})

print(len(dataset_files))
dataset_files[0]

## 3. What the network needs

For each patient the network expects:

| | Image | Mask |
|---|---|---|
| Type | PyTorch tensor | PyTorch tensor |
| Shape | (1, 256, 256): channel first, same size for everybody | (1, 256, 256) |
| Values | between 0 and 1 | 0 = background, 1 = lung |

Our files are 512 x 512, in Hounsfield units, with four labels. A **transform** is a function that does one step
of this preparation. MONAI has two versions of each transform:

- `LoadImage`, `Resize`, ... work on one array;
- `LoadImaged`, `Resized`, ... (with a final **d**) work on a **dictionary**. With `keys` we say which entries
  to transform. This way image and mask always go through the same steps.

We apply the transforms one at a time to the first patient and look at the result after each step.

In [ ]:
sample = dataset_files[0]
sample

## 4. Loading the files

`LoadImaged` replaces the paths with the arrays read from the files.

In [ ]:
load = LoadImaged(keys=["image", "label"])
sample = load(sample)

print("image:", type(sample["image"]).__name__, sample["image"].shape)
print("label:", type(sample["label"]).__name__, sample["label"].shape)

The result is a `MetaTensor`: a PyTorch tensor that also remembers the information of the file header.

## 5. Adding the channel

PyTorch wants the number of channels as the first dimension. A colour photo has 3 channels, a CT slice has 1.

In [ ]:
add_channel = EnsureChannelFirstd(keys=["image", "label"])
sample = add_channel(sample)

print("image:", sample["image"].shape)
print("label:", sample["label"].shape)

## 6. Keeping only the lungs

The mask contains 1 (right lung), 2 (left lung) and 3 (trachea). We want 1 for lung and 0 for everything else.
There is no ready transform for this, so we write a small function and apply it with `Lambdad`.

In [ ]:
def keep_lungs(label):
    return ((label == 1) | (label == 2)).float()

only_lungs = Lambdad(keys="label", func=keep_lungs)

print("values before:", np.unique(sample["label"]))
sample = only_lungs(sample)
print("values after :", np.unique(sample["label"]))

## 7. Rescaling the intensities

`ScaleIntensityRanged` does what we did by hand in notebook 01: it cuts the values to a window of Hounsfield
units (`a_min`, `a_max`) and rescales them (`b_min`, `b_max`). It is applied to the image only.

In [ ]:
scale = ScaleIntensityRanged(keys="image", a_min=-1000, a_max=400, b_min=0.0, b_max=1.0, clip=True)

print("before:", sample["image"].min().item(), sample["image"].max().item())
sample = scale(sample)
print("after :", sample["image"].min().item(), sample["image"].max().item())

## 8. Resizing

We reduce the images to 256 x 256: training is about four times faster and the lungs are still clearly visible.

`mode` chooses how the new pixels are computed. It is one for each key:

- `"bilinear"` for the image: each new pixel is an average of its neighbours;
- `"nearest"` for the mask: each new pixel copies the closest one. An average of 0 and 1 would give values such
  as 0.5, which are not labels.

In [ ]:
resize = Resized(keys=["image", "label"], spatial_size=(256, 256), mode=("bilinear", "nearest"))
sample = resize(sample)

print("image:", sample["image"].shape)
print("label:", sample["label"].shape, "values:", np.unique(sample["label"]))

In [ ]:
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title("image")
plt.imshow(sample["image"][0].T, cmap="gray")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.title("label")
plt.imshow(sample["label"][0].T, cmap="gray")
plt.axis("off")
plt.show()

`sample["image"][0]` takes the first (and only) channel, so that we have a 2D array to plot.
The image is already in the lung window: after the rescaling, 0 is -1000 HU and 1 is 400 HU.

## 9. Compose: all the steps together

`Compose` chains the transforms: the output of one is the input of the next. This is the object we will give
to the training code.

In [ ]:
preprocessing = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Lambdad(keys="label", func=keep_lungs),
    ScaleIntensityRanged(keys="image", a_min=-1000, a_max=400, b_min=0.0, b_max=1.0, clip=True),
    Resized(keys=["image", "label"], spatial_size=(256, 256), mode=("bilinear", "nearest")),
])

In [ ]:
# From the paths to the tensors, in one call
sample = preprocessing(dataset_files[1])

print("image:", sample["image"].shape, sample["image"].min().item(), sample["image"].max().item())
print("label:", sample["label"].shape, np.unique(sample["label"]))

## 10. Data augmentation

We have a few hundred images. To make the network more robust we show it, at every epoch, a slightly different
version of each image: a bit rotated, a bit larger or smaller, a bit shifted. This is **data augmentation**.

The transforms that start with `Rand` are random: each call gives a different result.
The same random change is applied to the image and to the mask, so they stay aligned.

In [ ]:
augmentation = RandAffined(
    keys=["image", "label"],
    prob=1.0,                        # probability of applying the transform
    rotate_range=0.2,                # rotation up to 0.2 radians (about 11 degrees)
    scale_range=0.1,                 # zoom in or out up to 10%
    translate_range=10,              # shift up to 10 pixels
    mode=("bilinear", "nearest"),    # as for the resize
    padding_mode="zeros",
)

In [ ]:
plt.figure(figsize=(16, 4))
for i in range(4):
    augmented = augmentation(sample)
    plt.subplot(1, 4, i + 1)
    plt.imshow(augmented["image"][0].T, cmap="gray")
    plt.imshow(np.ma.masked_equal(augmented["label"][0], 0).T, cmap="autumn", alpha=0.3)
    plt.axis("off")
plt.show()

The four pictures come from the same patient. The changes are small on purpose: the result must still look
like a real CT slice.

Augmentation is used **only on the training images**. Validation and test images are never modified,
otherwise the measured performance would change from one run to the next.

## 11. Exercises

Write your code in the cell below each exercise. Everything you need is in the sections above.

### Exercise 1 - Preprocess another patient

1. Apply `preprocessing` to the patient number 10 of `dataset_files` and save the result in `sample_10`.
2. Print the shape of the image and of the label.
3. Print the minimum and the maximum of the image.
4. Show the image and the label side by side.

In [ ]:
# 1. Apply the preprocessing
sample_10 = preprocessing(dataset_files[10])

# 2. Print the shapes
print(sample_10["image"].shape, sample_10["label"].shape)

# 3. Print minimum and maximum of the image
print(sample_10["image"].min().item(), sample_10["image"].max().item())

# 4. Show image and label
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(sample_10["image"][0].T, cmap="gray")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(sample_10["label"][0].T, cmap="gray")
plt.axis("off")
plt.show()

### Exercise 2 - A pipeline with smaller images

1. Write a new `Compose` called `preprocessing_128`, equal to `preprocessing` but with images of 128 x 128.
2. Apply it to the patient number 10 and save the result in `sample_128`.
3. Print the shape of the image: it must be (1, 128, 128).
4. Print the values of the label: they must be 0 and 1.

In [ ]:
# 1. The new pipeline
preprocessing_128 = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Lambdad(keys="label", func=keep_lungs),
    ScaleIntensityRanged(keys="image", a_min=-1000, a_max=400, b_min=0.0, b_max=1.0, clip=True),
    Resized(keys=["image", "label"], spatial_size=(128, 128), mode=("bilinear", "nearest")),
])

# 2. Apply it
sample_128 = preprocessing_128(dataset_files[10])

# 3. Shape of the image
print(sample_128["image"].shape)

# 4. Values of the label
print(np.unique(sample_128["label"]))

### Exercise 3 - Your own augmentation

`RandGaussianNoised` adds random noise to the image. It has no effect on the mask, so it is applied to the key
`"image"` only.

1. Create `noise = RandGaussianNoised(...)` with `keys="image"`, `prob=1.0` and `std=0.05`.
2. Apply `noise` to `sample_10` and save the result in `noisy`.
3. Show the original image and the noisy image side by side.
4. Change `std` to 0.2 and run again. Is the result still a realistic CT image?

In [ ]:
# 1. Create the transform
noise = RandGaussianNoised(keys="image", prob=1.0, std=0.05)

# 2. Apply it
noisy = noise(sample_10)

# 3. Show original and noisy
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title("original")
plt.imshow(sample_10["image"][0].T, cmap="gray")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.title("noisy")
plt.imshow(noisy["image"][0].T, cmap="gray")
plt.axis("off")
plt.show()

# 4. With std=0.2 the noise hides the small structures of the lung: too strong to be realistic.

### Extra - if you have finished

- **Left-right flip.** `RandFlipd(keys=["image", "label"], prob=1.0, spatial_axis=0)` mirrors the image.
  Apply it and show the result. The heart is now on the wrong side: would you use this augmentation?
  Would the answer change if the network had to tell the right lung from the left lung?
- **Wrong interpolation.** Build a pipeline without `Lambdad` and with `mode=("bilinear", "bilinear")`,
  then print `np.unique` of the label. How many different values do you get?

## 12. To go further

- MONAI transforms, the full list: https://docs.monai.io/en/stable/transforms.html
- MONAI tutorial, 2D transforms demo: https://github.com/Project-MONAI/tutorials/blob/main/modules/transforms_demo_2d.ipynb
- MONAI tutorial, loading medical images: https://github.com/Project-MONAI/tutorials/blob/main/modules/load_medical_images.ipynb
- MONAI in one notebook: https://github.com/Project-MONAI/tutorials/blob/main/2d_classification/monai_101.ipynb